In [2]:
import pandas as pd
import os

In [3]:
DATA_PATH = "../data"

files = os.listdir(DATA_PATH)
files

['fleet_cleaned_data.csv',
 'IMO1_GPS.csv',
 'IMO1_MOTIONS.csv',
 'IMO2_GPS.csv',
 'IMO2_MOTIONS.csv',
 'IMO3_GPS.csv',
 'IMO3_MOTIONS.csv']

In [4]:
gps1 = pd.read_csv("../data/IMO1_GPS.csv")
gps2 = pd.read_csv("../data/IMO2_GPS.csv")
gps3 = pd.read_csv("../data/IMO3_GPS.csv")

motions1 = pd.read_csv("../data/IMO1_MOTIONS.csv")
motions2 = pd.read_csv("../data/IMO2_MOTIONS.csv")
motions3 = pd.read_csv("../data/IMO3_MOTIONS.csv")

In [5]:
gps1.head(10)

,Timestamp,Course [deg] (NAVIGATION_GPS),Heading [deg] (NAVIGATION_GYRO),Latitude [deg] (NAVIGATION_GPS),Longitude [deg] (NAVIGATION_GPS),Speed [kn] (NAVIGATION_GPS)
0,2026-03-01 00:15:00,191.699997,120.120003,32.536419,-79.482132,0.589338
1,2026-03-01 00:30:00,81.900002,53.380001,32.532730,-79.477249,1.553984
2,2026-03-01 00:45:00,323.700012,323.790009,32.556839,-79.485764,6.874972
3,2026-03-01 01:00:00,314.700012,312.329987,32.580200,-79.509781,7.428957
4,2026-03-01 01:15:00,290.200012,292.010010,32.600632,-79.548882,9.383950
5,2026-03-01 01:30:00,298.000000,299.920013,32.618469,-79.591713,9.703033
6,2026-03-01 01:45:00,300.200012,301.140015,32.644550,-79.646019,12.613920
7,2026-03-01 02:00:00,298.200012,298.829987,32.674278,-79.708633,14.569470
8,2026-03-01 02:15:00,299.700012,299.950012,32.704220,-79.771370,14.597080
9,2026-03-01 02:30:00,299.700012,298.450012,32.731930,-79.828453,13.358700


In [6]:
# 2. Fonction de nettoyage et standardisation du schéma GPS
def clean_gps(df: pd.DataFrame, vessel_id: str) -> pd.DataFrame:
    # Standardisation dynamique des noms de colonnes (élimine la variabilité [NAVIGATION_GPS], etc.)
    rename_dict = {}
    for col in df.columns:
        c_lower = col.lower()
        if "timestamp" in c_lower:
            rename_dict[col] = "timestamp"
        elif "latitude" in c_lower:
            rename_dict[col] = "latitude"
        elif "longitude" in c_lower:
            rename_dict[col] = "longitude"
        elif "speed" in c_lower:
            rename_dict[col] = "speed"
        elif "course" in c_lower:
            rename_dict[col] = "course"
        elif "heading" in c_lower:
            rename_dict[col] = "heading"

    df_clean = df.rename(columns=rename_dict).copy()
    df_clean["vessel_id"] = vessel_id
    df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"])
    return df_clean

df_cleaned = clean_gps(gps1, "IMO1")
df_cleaned.head(10)

,timestamp,course,heading,latitude,longitude,speed,vessel_id
0,2026-03-01 00:15:00,191.699997,120.120003,32.536419,-79.482132,0.589338,IMO1
1,2026-03-01 00:30:00,81.900002,53.380001,32.532730,-79.477249,1.553984,IMO1
2,2026-03-01 00:45:00,323.700012,323.790009,32.556839,-79.485764,6.874972,IMO1
3,2026-03-01 01:00:00,314.700012,312.329987,32.580200,-79.509781,7.428957,IMO1
4,2026-03-01 01:15:00,290.200012,292.010010,32.600632,-79.548882,9.383950,IMO1
5,2026-03-01 01:30:00,298.000000,299.920013,32.618469,-79.591713,9.703033,IMO1
6,2026-03-01 01:45:00,300.200012,301.140015,32.644550,-79.646019,12.613920,IMO1
7,2026-03-01 02:00:00,298.200012,298.829987,32.674278,-79.708633,14.569470,IMO1
8,2026-03-01 02:15:00,299.700012,299.950012,32.704220,-79.771370,14.597080,IMO1
9,2026-03-01 02:30:00,299.700012,298.450012,32.731930,-79.828453,13.358700,IMO1


In [7]:
def clean_motions(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()
    # Renommer la colonne temporelle si elle s'appelle Timestamp
    for col in df_clean.columns:
        if col.lower() == "timestamp":
            df_clean = df_clean.rename(columns={col: "timestamp"})
            break
    df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"])
    return df_clean

In [8]:
# 4. Standardisation des DataFrames GPS et MOTIONS
gps_dict = {
    "IMO1": clean_gps(gps1, "IMO1"),
    "IMO2": clean_gps(gps2, "IMO2"),
    "IMO3": clean_gps(gps3, "IMO3"),
}

motions_dict = {
    "IMO1": clean_motions(motions1),
    "IMO2": clean_motions(motions2),
    "IMO3": clean_motions(motions3),
}

In [9]:
processed_dfs = []
for vessel_id in ["IMO1", "IMO2", "IMO3"]:
    df_gps = gps_dict[vessel_id].sort_values("timestamp").reset_index(drop=True)
    df_mot = (
        motions_dict[vessel_id].sort_values("timestamp").reset_index(drop=True)
    )

    # Imputation ciblée par ffill uniquement sur position/vitesse (ex: IMO1 à l'arrêt)
    cols_to_ffill = ["latitude", "longitude", "speed"]
    df_gps[cols_to_ffill] = df_gps[cols_to_ffill].ffill()

    # Fusion temporelle rigoureuse avec MOTIONS (jointure sur le timestamp le plus proche)
    df_merged = pd.merge_asof(
        df_gps,
        df_mot,
        on="timestamp",
        direction="nearest",
        suffixes=("", "_motion"),
    )

    processed_dfs.append(df_merged)

In [10]:
fleet_df = pd.concat(processed_dfs, ignore_index=True)

In [11]:
fleet_df

,timestamp,course,heading,latitude,longitude,speed,vessel_id,C1 [-] (Parametric Roll Advanced 2 to 1),C2 [-] (Parametric Roll Advanced 2 to 1),C3 [-] (Parametric Roll Advanced 2 to 1),...,X velocity [m/s] (Motion Sensor),Y acceleration [m/s2] (Motion Sensor),Y motion [m] (Motion Sensor),Y velocity [m/s] (Motion Sensor),Yaw acceleration [deg/s2] (Motion Sensor),Yaw motion [deg] (Motion Sensor),Yaw velocity [deg/s] (Motion Sensor),Z acceleration [m/s2] (Motion Sensor),Z motion [m] (Motion Sensor),Z velocity [m/s] (Motion Sensor)
0,2026-03-01 00:15:00,191.699997,120.120003,32.536419,-79.482132,0.589338,IMO1,0.040320,0.189111,0.349515,...,0.051748,0.063235,0.125345,0.084968,0.325145,0.155002,0.166379,0.026153,0.062072,0.038889
1,2026-03-01 00:30:00,81.900002,53.380001,32.532730,-79.477249,1.553984,IMO1,0.017048,0.227061,0.371903,...,0.053966,0.083662,0.129813,0.092231,0.317784,0.148082,0.160054,0.034064,0.080624,0.050374
2,2026-03-01 00:45:00,323.700012,323.790009,32.556839,-79.485764,6.874972,IMO1,0.051061,0.152411,0.388419,...,0.076025,0.162496,0.243317,0.163351,0.323918,0.161318,0.171795,0.113981,0.252662,0.163770
3,2026-03-01 01:00:00,314.700012,312.329987,32.580200,-79.509781,7.428957,IMO1,0.037836,0.165556,0.272704,...,0.061847,0.145865,0.217470,0.140468,0.319919,0.158526,0.169841,0.079326,0.196039,0.121064
4,2026-03-01 01:15:00,290.200012,292.010010,32.600632,-79.548882,9.383950,IMO1,0.038369,0.213286,0.513634,...,0.048731,0.146882,0.129693,0.087340,0.324215,0.149111,0.168763,0.040560,0.113241,0.065879
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43445,2026-08-01 22:45:00,121.250000,124.019997,5.048659,98.277397,12.038440,IMO3,0.059327,0.079094,0.459717,...,0.037688,0.030917,0.061146,0.043105,0.292424,0.129372,0.142921,0.016070,0.041220,0.024682
43446,2026-08-01 23:00:00,121.169998,123.839996,5.023224,98.319267,11.741300,IMO3,0.033157,0.124861,0.356598,...,0.037810,0.029747,0.059757,0.041906,0.280154,0.125642,0.139048,0.015166,0.035331,0.021795
43447,2026-08-01 23:15:00,121.010002,124.209999,4.997867,98.360878,11.648320,IMO3,0.023052,0.142889,0.384881,...,0.038994,0.026594,0.055107,0.039372,0.277833,0.119801,0.135772,0.014950,0.037284,0.022741
43448,2026-08-01 23:30:00,124.059998,125.989998,4.970927,98.401360,11.639700,IMO3,0.024439,0.062875,0.300265,...,0.038974,0.026475,0.056380,0.040660,0.284036,0.123396,0.138653,0.013056,0.032165,0.019450


In [12]:
# 7. Calcul des variables métier imposées par le sujet (Feature Engineering)
# RPM = 4 * SOG (Speed)
fleet_df["rpm"] = 4 * fleet_df["speed"]

# Consommation (tonnes / jour à mer calme) = 150 * (SOG / 15)^3
fleet_df["fuel_consumption_t_per_day"] = 150 * ((fleet_df["speed"] / 15) ** 3)

# Consommation horaire (tonnes / heure)
fleet_df["fuel_consumption_t_per_hour"] = (
    fleet_df["fuel_consumption_t_per_day"] / 24
)

# Coût carburant ($ / heure) basé sur 1 tonne = $1 000 USD
fleet_df["fuel_cost_usd_per_hour"] = (
    fleet_df["fuel_consumption_t_per_hour"] * 1000
)

# 8. Nettoyage final des colonnes doublons éventuelles
fleet_df = fleet_df.loc[:, ~fleet_df.columns.duplicated()]



In [13]:

fleet_df.to_csv("../data/fleet_cleaned_data.csv", index=False)


In [14]:
fleet_df.columns

Index(['timestamp', 'course', 'heading', 'latitude', 'longitude', 'speed',
       'vessel_id', 'C1 [-] (Parametric Roll Advanced 2 to 1)',
       'C2 [-] (Parametric Roll Advanced 2 to 1)',
       'C3 [-] (Parametric Roll Advanced 2 to 1)',
       'CTotal [-] (Parametric Roll Advanced 2 to 1)',
       'Parametric Roll Advanced 2 to 1 [-] (Parametric Roll Advanced 2 to 1)',
       'Pitch acceleration [deg/s2] (Motion Sensor)',
       'Pitch motion [deg] (Motion Sensor)',
       'Pitch motion - period [sec] (Motion Sensor)',
       'Pitch velocity [deg/s] (Motion Sensor)',
       'Roll acceleration [deg/s2] (Motion Sensor)',
       'Roll motion [deg] (Motion Sensor)',
       'Roll motion - period [sec] (Motion Sensor)',
       'Roll velocity [deg/s] (Motion Sensor)',
       'X acceleration [m/s2] (Motion Sensor)', 'X motion [m] (Motion Sensor)',
       'X velocity [m/s] (Motion Sensor)',
       'Y acceleration [m/s2] (Motion Sensor)', 'Y motion [m] (Motion Sensor)',
       'Y velocity [m